In [ ]:
#| default_exp thin

In [ ]:
#| hide
from fastcore.test import *

A thin app is a small native launcher. It owns the windows, and it runs the app's Python server as its child. The Python packages do not ship in the app: they come from a signed environment lock that has its own number and its own channel. `contract/api-1.md` in this repository gives the contract between the launcher, the server and the lock tools.

This module describes one thin app with `Thin`. Every name the launcher, the lock tools and the server use comes from it: the support folder, the env marker, the signing namespace, the `<PREFIX>_*` variables, the bridge handler. `Thin.config` writes `kavacha.json`, which the launcher reads in place of constants. `Thin.info_plist` writes the bundle's `Info.plist`, and `childenv` gives the clean child environment.

In [ ]:
#| export
from __future__ import annotations
import json, os, sys, tomllib
from dataclasses import dataclass, field, fields, asdict
from fastcore.all import Path

### The child environment

`CHILDENV` is the environment spec that the launcher and the lock tools both read. The common keys apply to every OS; a key under `darwin`, `linux` or `win32` replaces the common one on that OS. `{prefix}` is the app's variable prefix, such as `LEELA_`.

In [ ]:
#| export
CHILDENV = dict(api=1,
    drop=['VIRTUAL_ENV', 'CONDA_PREFIX', 'PYTHONHOME', 'PYTHONPATH', 'PYTHONSTARTUP', 'PYTHONUSERBASE', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__'],
    drop_prefixes=['UV_', '{prefix}HOST_', '{prefix}LOCK_', '{prefix}ENV_', '{prefix}APP_'],
    set={'LANG': 'en_US.UTF-8', 'NO_COLOR': '1'},
    uv={'UV_NO_CONFIG': '1', 'UV_CACHE_DIR': '{cache}', 'UV_PYTHON_INSTALL_DIR': '{python}', 'UV_PYTHON_PREFERENCE': 'only-managed',
        'UV_PYTHON': '{python_version}', 'UV_PROJECT_ENVIRONMENT': '{env}', 'UV_FROZEN': '1', 'UV_NO_PROGRESS': '1', 'UV_LINK_MODE': 'clone'},
    server={'PYTHONUNBUFFERED': '1'},
    tool_dirs=[],
    sync=['sync', '--frozen', '--no-install-project', '--project', '{lock}'],
    extra=['--extra', '{name}'], inexact=['--inexact'],
    server_args=['--port', '{port}'],
    darwin=dict(tool_dirs=['~/.local/bin', '/opt/homebrew/bin', '/opt/homebrew/sbin', '/usr/local/bin', '/usr/bin', '/bin', '/usr/sbin', '/sbin']),
    linux=dict(tool_dirs=['~/.local/bin', '/usr/local/bin', '/usr/bin', '/bin', '/usr/sbin', '/sbin'], uv={'UV_LINK_MODE': 'hardlink'}),
    win32=dict(tool_dirs=['~/.local/bin', '~/AppData/Local/Programs/Python/Launcher'], uv={'UV_LINK_MODE': 'copy'}))
OSES = ('darwin', 'linux', 'win32')

`childenv` flattens the spec for one OS and fills in the prefix. The launcher's bundle holds the flat form, so the Swift and Rust sides read one shape.

In [ ]:
#| export
def childenv(prefix='APP_', platform=None, server_args=None, base=None):
    "The flat child environment spec for `platform`, with `prefix` filled in."
    s, plat = json.loads(json.dumps(base or CHILDENV)), platform or sys.platform
    over = s.pop(plat, {}) if plat in OSES else {}
    for o in OSES: s.pop(o, None)
    for k, v in over.items(): s[k] = s[k] | v if isinstance(v, dict) and isinstance(s.get(k), dict) else v
    s['drop_prefixes'] = [p.replace('{prefix}', prefix) for p in s['drop_prefixes']]
    if server_args is not None: s['server_args'] = list(server_args)
    return s

In [ ]:
s = childenv('LEELA_', 'darwin')
test_eq(s['drop_prefixes'], ['UV_', 'LEELA_HOST_', 'LEELA_LOCK_', 'LEELA_ENV_', 'LEELA_APP_'])
test_eq(s['tool_dirs'][:2], ['~/.local/bin', '/opt/homebrew/bin'])
test_eq('darwin' in s or 'win32' in s, False)
w = childenv('X_', 'win32')
test_eq((w['uv']['UV_LINK_MODE'], w['uv']['UV_NO_CONFIG']), ('copy', '1'))

### The spec

`Thin` is a flat dataclass. A field left empty is derived from `name`, so an app gives only what differs. The groups are: identity, the server, lock signing, the lock, the native capabilities and the distribution.

In [ ]:
#| export
@dataclass
class Thin:
    "One thin app: identity, server, lock signing, lock, native capabilities and distribution."
    name: str
    identifier: str = ''
    package: str = ''                 # the Python distribution the lock pins; its console script is the server
    version: str = '0.0.0'
    build: str = ''
    summary: str = ''
    copyright: str = ''
    category: str = 'public.app-category.developer-tools'
    repo: str = ''                    # `owner/name` on GitHub: the lock channel and the app releases
    # the server
    server: str = ''                  # the console script in the env's bin
    server_args: list = field(default_factory=lambda: ['--port', '{port}'])
    health: str = '/health'
    env_prefix: str = ''
    api: int = 1
    # lock signing
    sig_ns: str = ''
    marker: str = ''
    keys: str = 'packaging/lock/keys.pub'
    key_names: list = field(default_factory=list)
    # the lock
    lock_dir: str = 'packaging/lock'
    lock_project: str = ''
    python: str = '3.13.18'
    platforms: list = field(default_factory=lambda: ['darwin-arm64'])
    macos_min: str = '14.0'
    extras: list = field(default_factory=list)
    seed_extra: str = ''              # the extra `lock test` adds to a copied env
    owners: list = field(default_factory=list)   # author names whose packages skip the cooldown
    owner_org: str = ''               # where `lock diff` looks for an owner's tags
    wheels: dict = field(default_factory=dict)   # {name: {version, url, sha256}}: a package pinned to one wheel, allowed only in an extra
    source_only: list = field(default_factory=list)
    imports: list = field(default_factory=list)  # what `lock test` imports in the env
    cooldown: int = 7
    record_days: int = 7
    hooks: str = ''                   # a Python file with the app's `extra_reqs()` and `smoke(s)`
    childenv: str = ''                # a JSON file in place of kavacha's `CHILDENV`
    uv_version: str = 'packaging/uv.version'
    build_dir: str = 'build/thin'
    uv_sha256: str = 'packaging/uv.sha256'
    # native capabilities
    bridge: list = field(default_factory=lambda: ['open_folder', 'pick_folder', 'pick_path', 'open_path', 'recent', 'set_background', 'set_title', 'titlebar'])
    bridge_global: str = 'pywebview'  # the page reads `window.<bridge_global>.api`
    sites: dict = field(default_factory=dict)    # {list, go}: a window per site on a port of its own
    routes: dict = field(default_factory=dict)   # {open, grant, menu, recent}: the server routes the launcher calls
    capabilities: list = field(default_factory=lambda: ['confirm', 'grant', 'update', 'extras'])
    menus: str = '/app/menu.json'
    menu_action_js: str = 'leeAct({action})'
    window: dict = field(default_factory=dict)   # {size, min_size, home, title_prefix}
    titlebar: str = 'native'          # `mac-unified`: a transparent, unified title bar the page draws into; `native` elsewhere
    doc_types: list = field(default_factory=list)
    imported_types: list = field(default_factory=list)
    url_schemes: list = field(default_factory=list)
    usage: dict = field(default_factory=dict)    # {NSMicrophoneUsageDescription: ...}
    entitlements: dict = field(default_factory=dict)
    icon: str = ''
    # distribution
    cask: str = ''
    cask_template: str = ''          # a cask file whose `version` and `sha256` lines are filled in
    tap: str = ''
    homepage: str = ''
    winget: str = ''
    launcher: str = ''                # the kavacha launcher release, such as `launcher-v1`
    launcher_sha256: str = ''
    max_mb: int = 60
    max_zip_mb: int = 25

    def __post_init__(self):
        low = self.name.lower()
        for k, v in dict(identifier=f'org.{low}.{low}', package=low, server=low, env_prefix=f'{self.name.upper()}_', sig_ns=f'{low}-lock',
                         marker=f'.{low}-env.json', lock_project=f'{low}-lock', cask=low).items():
            if not getattr(self, k): setattr(self, k, v)
        self.key_names = self.key_names or [f'{low}_lock_ed25519', f'{low}_lock_emergency_ed25519']
        self.window = dict(size=[1440, 900], min_size=[900, 560], home=f'{low}:home', title_prefix=f'{low} — ') | self.window
        self.routes = dict(open='/hub/open', grant='/hub/grant', recent='/app/recent') | self.routes
        self.owner_org = self.owner_org or self.repo.split('/')[0]
        self.seed_extra = self.seed_extra or (self.extras[-1] if self.extras else '')

`Thin.load` reads the spec from `[tool.kavacha.thin]` in `pyproject.toml`, or from `packaging/kavacha.toml`. A TOML table such as `[lock]` only groups its keys: `lock.python` is the field `python`. A table whose name is also a field, such as `[sites]`, stays a table.

In [ ]:
#| export
def _flat(d):
    names, out = {f.name for f in fields(Thin)}, {}
    for k, v in d.items():
        if isinstance(v, dict) and k not in names: out |= _flat(v)
        else: out[k] = v
    return out

@classmethod
def _load(cls, root='.'):
    "The spec of the app in `root`, from `[tool.kavacha.thin]` in pyproject.toml or packaging/kavacha.toml."
    root = Path(root)
    if (f := root/'packaging'/'kavacha.toml').is_file(): d = tomllib.loads(f.read_text())
    else: d = tomllib.loads((root/'pyproject.toml').read_text()).get('tool', {}).get('kavacha', {}).get('thin', {})
    if not d: raise SystemExit(f'no thin app spec in {root}: add packaging/kavacha.toml or [tool.kavacha.thin]')
    t = cls(**_flat(d)); t.root = root.resolve()
    return t
Thin.load = _load

In [ ]:
import tempfile
d = Path(tempfile.mkdtemp())
(d/'packaging').mkdir(); (d/'packaging'/'kavacha.toml').write_text('name = "Demo"\nrepo = "me/demo"\n[lock]\npython = "3.12.9"\nextras = ["a", "b"]\n[sites]\nlist = "/s"\n')
t = Thin.load(d)
test_eq((t.python, t.extras, t.sites, t.seed_extra), ('3.12.9', ['a', 'b'], {'list': '/s'}, 'b'))
test_eq((t.identifier, t.sig_ns, t.marker, t.env_prefix, t.cask, t.owner_org), ('org.demo.demo', 'demo-lock', '.demo-env.json', 'DEMO_', 'demo', 'me'))

### What the launcher reads

`Thin.config` is `kavacha.json`, the launcher's only source of names. A launcher that finds no such file, or one with a higher `api` than its own, refuses to start.

In [ ]:
#| export
def _config(self):
    "kavacha.json: every name the launcher uses, from this spec."
    return dict(api=self.api, name=self.name, identifier=self.identifier, support=self.name, marker=self.marker, sig_ns=self.sig_ns,
                env_prefix=self.env_prefix, server=self.server, health=self.health, lock_project=self.lock_project, python=self.python.rsplit('.', 1)[0],
                channel=f'https://github.com/{self.repo}/releases/download' if self.repo else '', extras=list(self.extras),
                bridge=dict(handler=self.name.lower(), flag=f'__{self.name.lower()}', methods=list(self.bridge), js_global=self.bridge_global),
                sites=dict(self.sites), routes=dict(self.routes), capabilities=list(self.capabilities), menus=self.menus, menu_action_js=self.menu_action_js,
                window=dict(self.window), titlebar=dict(style=self.titlebar, query='titlebar', drag_class='pywebview-drag-region'), cask=self.cask, debug_env=f'{self.env_prefix}WEBVIEW_DEBUG')
Thin.config = _config

`Thin.info_plist` is the bundle's `Info.plist`. The launcher binary is always `MacOS/<name>`, and the icon is `Resources/AppIcon.icns`.

In [ ]:
#| export
def _plist(self):
    "The bundle's Info.plist entries."
    p = dict(CFBundleDisplayName=self.name, CFBundleExecutable=self.name, CFBundleIconFile='AppIcon', CFBundleIdentifier=self.identifier,
             CFBundleName=self.name, CFBundlePackageType='APPL', CFBundleShortVersionString=self.version, CFBundleVersion=self.build or self.version,
             LSApplicationCategoryType=self.category, LSMinimumSystemVersion=self.macos_min, NSAppTransportSecurity={'NSAllowsLocalNetworking': True},
             NSHighResolutionCapable=True)
    if self.copyright: p['NSHumanReadableCopyright'] = self.copyright
    if self.doc_types: p['CFBundleDocumentTypes'] = self.doc_types
    if self.imported_types: p['UTImportedTypeDeclarations'] = self.imported_types
    if self.url_schemes: p['CFBundleURLTypes'] = [dict(CFBundleURLName=f'{self.identifier}.{s}', CFBundleURLSchemes=[s]) for s in self.url_schemes]
    return p | dict(self.usage)
Thin.info_plist = _plist

In [ ]:
t = Thin('Demo', version='1.0.0', build='4', url_schemes=['demo'], usage={'NSMicrophoneUsageDescription': 'to listen'})
p = t.info_plist()
test_eq((p['CFBundleExecutable'], p['CFBundleVersion'], p['CFBundleURLTypes'][0]['CFBundleURLSchemes']), ('Demo', '4', ['demo']))
test_eq(p['NSMicrophoneUsageDescription'], 'to listen')
c = t.config()
test_eq((c['support'], c['bridge']['handler'], c['bridge']['flag'], c['debug_env'], c['python']), ('Demo', 'demo', '__demo', 'DEMO_WEBVIEW_DEBUG', '3.13'))

`Thin.spec_env` is the flat child environment for this app: the app's own JSON file when it gives one, else kavacha's with its prefix and server arguments.

In [ ]:
#| export
def _spec_env(self, platform=None):
    "The flat child environment spec of this app."
    f = self.childenv and Path(getattr(self, 'root', '.'))/self.childenv
    if f and f.is_file(): return json.loads(f.read_text())
    return childenv(self.env_prefix, platform, self.server_args)
Thin.spec_env = _spec_env

In [ ]:
test_eq(Thin('Demo', server_args=['--app', '--port', '{port}']).spec_env('darwin')['server_args'], ['--app', '--port', '{port}'])

`Thin.write` puts the three generated files in a folder: `kavacha.json`, `childenv.json` and `Info.plist`.

In [ ]:
#| export
def _write(self, out, platform='darwin', plist=None):
    "Write kavacha.json, childenv.json and Info.plist (with `plist` on top) into `out`."
    import plistlib
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    (out/'kavacha.json').write_text(json.dumps(self.config(), indent=1, sort_keys=True) + '\n')
    (out/'childenv.json').write_text(json.dumps(self.spec_env(platform), indent=2) + '\n')
    (out/'Info.plist').write_bytes(plistlib.dumps(self.info_plist() | dict(plist or {})))
    return out
Thin.write = _write

In [ ]:
o = Thin('Demo').write(Path(tempfile.mkdtemp()))
test_eq(sorted(f.name for f in o.iterdir()), ['Info.plist', 'childenv.json', 'kavacha.json'])

### The command line

`kavacha lock …` is the lock tool (`kavacha.lock`), and `kavacha ship mac …` builds and ships the app (`kavacha.ship`).

In [ ]:
#| export
def cli(argv=None):
    "kavacha lock … | kavacha ship …"
    argv = list(sys.argv[1:] if argv is None else argv)
    if not argv or argv[0] not in ('lock', 'ship'): raise SystemExit('usage: kavacha lock <command> | kavacha ship mac')
    if argv[0] == 'lock': from kavacha.lock import main
    else: from kavacha.ship import main
    return main(argv[1:])